In [ ]:
import os, re
import numpy as np
import pandas as pd
from scipy import integrate
import pytz
import math

Functions below are a direct translation of matlab code provided for ROZE data processing. Some differences between the two might arise from the interpolation used in lagCovFFT.

# Functions

In [ ]:
def windrot (u_o, v_o, w_o, nRot):
    u1= u_o
    v1= v_o
    w1= w_o
    
    u1m = np.mean(u1) 
    v1m = np.mean(v1) 
    w1m = np.mean(w1)

    u1m2 = u1m ** 2 
    v1m2 = v1m ** 2 
    w1m2 = w1m ** 2

    rms_uv = np.sqrt(u1m2+v1m2) 
    rms_uvw = np.sqrt(u1m2+v1m2+w1m2)
    
    CE = u1m/rms_uv
    SE = v1m/rms_uv
    CT = rms_uv/rms_uvw
    ST = w1m/rms_uvw
    
    #first two rotations
    u2 = (u1*CT*CE) + (v1*CT*SE) + (w1*ST)
    v2 = (v1*CE) - (u1*SE)
    w2 = (w1*CT) - (u1*ST*CE) - (v1*ST*SE)
    
    #third rotation 
    v2p = v2 - np.mean(v2) 
    w2p = w2 - np.mean(w2)

    B = 0.5 * np.arctan(2 * np.mean(v2p*w2p)/(np.mean(v2p ** 2) - np.mean(w2p**2)))
    CB = np.cos(B)
    SB = np.sin(B)

    v3 = (v2*CB) + (w2*SB)
    w3 = (w2*CB) - (v2*SB)
    
    if options['nRot'] == 1:
        u_r = u1
        v_r = v1
        w_r = w2
    if options['nRot'] == 2:
        u_r = u2
        v_r = v2
        w_r = w2
    if options['nRot'] == 3:
        u_r = u2
        v_r = v3
        w_r = w3
        
    #rotation angles
    eta   = np.arcsin(SE) * (180/np.pi)
    theta = np.arcsin(ST) * (180/np.pi)
    beta  = B * (180/np.pi)
    
    #returns a list of individual rotated wind vectors and a list of angles
    return([u_r, v_r, w_r, [eta, theta, beta]])

In [ ]:
def despike (x, q = 7, fill = 'nan'):
    
    #Find median
    m = np.nanmedian(x)
    
    #median absolute deviation
    MAD = np.nanmedian(abs(x - m))
    
    #find rows that meet the 'spike' criteria
    spike = abs(x - m) >= q * MAD/0.6745

    spikePct = (sum(spike) / len(x)) * 100
    
    #if no spikes
    if spikePct == 0:
        return([x, spikePct, spike])
    
    #Fill in gaps from despiking
    x[spike] = np.nan

    if fill == 'interp':
        x = x.interpolate(method = 'linear')
        
    #returns a list of despiked x, spike percent, and spike flags (indices that mark spike in x)
    return([x, spikePct, spike])

In [ ]:
def detrend (x, t, trendType = 'linear', frameSize = 100):
    
    #calculate trends
    xmean = np.mean(x) #mean

    fit = np.polyfit(t[~np.isnan(x)], x[~np.isnan(x)], 1)
    xlin = np.polyval(fit, t[~np.isnan(x)]) #linear
    xsmooth = x.rolling(frameSize, min_periods = 1).mean()

    if trendType == 'mean':
        x_dt = x - xmean
    if trendType == 'linear':
        x_dt = x - xlin
    if trendType == 'smooth':
        x_dt = x - xsmooth
        
    #returns a list of detrended x and all three trends 
    return([x_dt, xmean, xlin, xsmooth])

In [ ]:
def lagCovFFT (x, y, Fcut = 0, Fsample = 0, maxlag = 0):
    
    #match na: mark y as na where x is na, and viceversa
    nay = np.isnan(y)
    nax = np.isnan(x)

    x[nay] = np.nan
    y[nax] = np.nan
    
    #eliminate na
    x = x.loc[~x.isna()]
    y = y.loc[~y.isna()]

    N = len(y)

    if maxlag == 0:
        maxlag = int(N/2)

    if Fcut == 0:
        Fcut = []

    #Interpolate to gap-fill
    x = x.interpolate(method = 'nearest', limit = 20000, limit_direction = 'both').fillna(method = 'bfill').fillna(method = 'ffill')
    y = y.interpolate(method = 'nearest', limit = 20000, limit_direction = 'both').fillna(method = 'bfill').fillna(method = 'ffill')

    x = x - np.mean(x)
    y = y - np.mean(y)

    lags = np.array([n for n in range(int(-maxlag), int(maxlag) + 1)])
    nfft = 2 ** math.ceil(math.log2(2 * N - 1)) 

    xfft = np.fft.fft(x,nfft)
    yfft = np.fft.fft(y,nfft)
    xyCr = xfft * np.conj(yfft) #cross-spectrum

    if Fcut != 0 and Fsample != 0: 
        df = Fsample/nfft 
        icut = math.floor(Fcut/df) 
        icut = np.concatenate([[n for n in range(1, icut+1)],
                               [n for n in range (nfft-icut, nfft)] #xyCr indices start at 0
                              ]) 
        xyCr[icut]=0 #snip


    #inverse fft to get lag covariance
    sigxy = np.fft.ifft(xyCr)
    sigxy = np.concatenate([sigxy[len(sigxy) - maxlag :], sigxy[0:maxlag + 1]], axis = 0) #rearrange
    nlag = [N - abs(i) for i in lags]
    sigxy = sigxy/nlag

    #Function returns list of cov of x and y at each lag and lags
    return([sigxy, lags])

In [ ]:
def pspectra (Fs, x):
    #Nans to 0
    x[np.isnan(x)] = 0
    
    x = x - np.mean(x)
    
    N = len(x)
    df = Fs/N
    Ny = np.floor(N/2) 
    f = [n for n in np.arange(0, Fs + df, df)]
    f = np.array(f[1: int(Ny)])
    
    xx = np.fft.fft (x, N)
    pxx = xx * np.conj(xx)/ (N**2)
    psdx = 2 * pxx[1:int(Ny)]/df
    
    #Comapre vairance from power spectrum density and data
    varx = np.std(x) ** 2
    var2x = sum(psdx) * df #psd variance
    if abs(varx - var2x)/ varx > .01:
        print('Relative differences between calculated and psd-estimated variance is greater than 0.01.')
    
    #normalize psd by variance
    psdxnorm = psdx/var2x
    
    return ([f, psdx, psdxnorm])

In [ ]:
def ogive (f, s, absFlag = False):
    N = len(f)
    
    #Nans to 0
    s[np.isnan(s)] = 0
     
    fflip = False
    if f[0] > f[1]:
        f = np.flipud(f)
        s = np.flipud(s)
        fflip = True
    
    #Integrate
    og = integrate.cumtrapz(s, f, initial = 0) 
    
    #Normalize 
    tot = np.tile(og[-1:],(N,1)).ravel()
    og = np.divide(og, tot)

    #flip back if needed
    if fflip:
        og = np.flipud(og)
        
    return(og)

In [ ]:
def cospectra (Fs, x, y):
    #match na: mark y as na where x is na, and viceversa
    nay = np.isnan(y)
    nax = np.isnan(x)

    x[nay] = np.nan
    y[nax] = np.nan
    
    #Nans to 0
    x[np.isnan(x)] = 0
    y[np.isnan(y)] = 0
    
    x = x - np.mean(x)
    y = y - np.mean(y)
    
    N = len(x)
    df = Fs/N
    Ny = np.floor(N/2) #Nyquist frequency
    f = [n for n in np.arange(0, Fs + df, df)]
    f = np.array(f[1: int(Ny)])
    
    #PSD of x and y (gas and wind)
    xx = np.fft.fft (x, N)
    pxx = xx * np.conj(xx)/ (N**2)
    psdx = 2 * pxx[1:int(Ny)]/ df

    yy = np.fft.fft (y, N)
    pyy  = yy * np.conj(yy)/ (N**2)
    psdy = 2 * pyy[1:int(Ny)]/ df
    
    #cospectra
    cross = xx[1:int(Ny)] * np.conj(yy[1:int(Ny)])/ (N**2 * df)
    
    co = 2 * np.real(cross) #real part
    qu = 2 * np.imag(cross) #imaginary part
    
    #Compare covariance from cospectra and data
    covxy = np.mean(x * y)
    cov2xy = sum(co) * df
    
    if abs((covxy-cov2xy) / covxy) > 0.01:
        print ('Relative differences between calculated and cospectra-estimated covariance is greater than 0.01.')
    
    #Normalize
    co_norm = co/abs(cov2xy)
    qu_norm = qu/abs(sum(qu) * df)
    
    return( [f, co, co_norm, qu, qu_norm] )

In [ ]:
def logbin (xdata, ydata, numint):
    
    logx = np.log(xdata)

    #mark inf as na
    logx[np.isinf(logx)] = 0

    #Get interval
    logmin = np.min(logx)
    logmax = np.max(logx)
    interval = abs(logmax - logmin) / numint

    intvec = np.exp(logmin + interval * np.array([n for n in range(0, numint+1)]))

    #Create empty vectors to fill with means in interval
    newx = np.empty(numint)
    newy = np.empty(numint)

    for j in range(0, numint):
        maskx = (xdata >= intvec[j]) & (xdata < intvec[j] + intvec[j+1])
        newx[j] = np.mean(xdata[maskx])
        newy[j] = np.mean(ydata[maskx])
        
    return (newx, newy)